# B12 Final External-Test Ablation and Evaluation

This notebook contains the supplied `VISUAL_ONLY` and `VISUAL_META` external-test pipeline. The experiment logic, model settings, calibration procedure, ensemble-weight search, decision rule, and reporting functions are preserved from the supplied source. The changes made for this GitHub copy are limited to English-language comments/messages and configurable, non-personal file paths.

## Before running

Set `SKIN_LESION_PROJECT_ROOT` to the directory containing the following structure, or edit the path configuration cell:

```text
<project-root>/
├── splits/
│   ├── train_split.csv
│   └── val_split.csv
├── features/
│   ├── swin/ft768/
│   ├── eva02/
│   └── coatnet/ft672/
├── metadata/ISIC_2019_Training_Metadata.csv
├── external_test/ISIC_2019_Test_Cleaned.csv
├── external_features/
│   ├── swin/swin_isic2019_test_features.npy
│   ├── eva02/eva_isic2019_test_features.npy
│   └── coatnet/coat_isic2019_test_features.npy
└── outputs/
```

The external-test CSV must contain the image identifier and labels expected by the loader. For `VISUAL_META`, it also needs the age/anatomical-region fields used by the metadata loader (and, where available, valid lesion identifiers). If these fields are absent, this implementation will substitute missing metadata values; verify the source CSV before interpreting metadata-fusion results.

The notebook contains no saved execution outputs. Large datasets, feature matrices, and generated results should not be committed to the repository.


## Imports and deterministic backend configuration

Imports the required libraries and configures deterministic PyTorch/CUDA behavior.

In [ ]:
# Final model pipeline
# -*- coding: utf-8 -*-
# ============================================================
#  B12_FINAL_EXTERNAL_TEST_STRICT_ABLATION.py
#  - Pipeline     : StratifiedGroupKFold (Train+Val only) -> OOF Fold-Ensembling
#  - Calibration  : Two-Stage (T_i step=0.02, T_ens step=0.005)
#  - Fusion       : Strict NLL-optimal Weight Search
#  - Decision     : Standard Argmax vs OOF BAcc-Optimized Rule
#  - Ablation     : VISUAL_ONLY vs. VISUAL_META (Metadata Fusion)
#  - Reproducibility: STRICT DETERMINISM
#  - Deliverables : 95% CI Bootstrapped Metrics & Class-wise Reports (inc. ROC-AUC)
# ============================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, platform, json, warnings, copy
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, log_loss,
    roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_fscore_support
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# --- STRICT REPRODUCIBILITY ---
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


## Project configuration

Defines data locations, output paths, model settings, split protocol, and fixed experiment parameters. Update the project root for your environment.

In [ ]:

# -------------------- Configuration --------------------
# Set SKIN_LESION_PROJECT_ROOT to your project directory. Expected subfolders are documented above.
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "splits")
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin" / "ft768"),
    "EVA" : str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet" / "ft672"),
}

META_CSV = str(PROJECT_ROOT / "metadata" / "ISIC_2019_Training_Metadata.csv")
EXT_CSV = str(PROJECT_ROOT / "external_test" / "ISIC_2019_Test_Cleaned.csv")
EXT_FEATURES = {
    "SWIN": str(PROJECT_ROOT / "external_features" / "swin" / "swin_isic2019_test_features.npy"),
    "EVA" : str(PROJECT_ROOT / "external_features" / "eva02" / "eva_isic2019_test_features.npy"),
    "COAT": str(PROJECT_ROOT / "external_features" / "coatnet" / "coat_isic2019_test_features.npy"),
}

BASE_SAVE_DIR = str(PROJECT_ROOT / "outputs" / "b12_external_final_publication_outputs")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
N_SPLITS = 5

EXPERIMENT_MODES = ["VISUAL_ONLY", "VISUAL_META"]

USE_PCA = True
PCA_VARIANCE = 0.99
T_MIN, T_MAX, T_STEP = 0.70, 1.50, 0.02
STAGE2_T_STEP = 0.005
MLP_BATCH = 256
COAT_FLOOR = 0.05
W_STEP = 0.02
N_CALIB_BINS = 15
N_BOOTSTRAP = 1000

CANONICAL_CLASS_NAMES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]
CANONICAL_CLASS_MAP = {name: idx for idx, name in enumerate(CANONICAL_CLASS_NAMES)}

CHAMPION_AGE_CFG = {"edges": [(0,10),(10,20),(20,30),(30,40),(40,50),(50,60),(60,70),(70,80),(80,np.inf)], "add_missing": False}

NAMED_PRESETS = {
    "SwinMLP": {"hidden":[2048], "dropout":0.1605757337, "lr":3.41409509895e-4, "weight_decay":1.488348195284e-4, "epochs":70},
    "EvaMLP":  {"hidden":[2048], "dropout":0.3203382207442766, "lr":9.765455593461e-4, "weight_decay":8.212522451000109e-05, "epochs":70},
}
HP_LOGREG = {"C":2.3402196063650043, "max_iter":587, "tol":3.6e-04, "solver":"lbfgs", "class_weight":"balanced", "n_jobs":-1, "random_state":RANDOM_STATE}
COAT_C_GRID = [1.6, HP_LOGREG["C"], 3.2]


## Random-seed configuration

Defines the random-seed routine used by the experiment.

In [ ]:

def set_seed():
    import random
    random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE); torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)


## Data, split, and metadata loaders

Loads the internal train/validation data and official external-test data, then prepares image identifiers and metadata mappings.

In [ ]:

# -------------------- Data & Metadata Loaders --------------------
def _glob_any(p: Path, names):
    for nm in names:
        if (p / nm).exists(): return str(p / nm)
    return None

def _normalize_ids(arr):
    return pd.Series(arr, dtype=str).str.replace(r"^.*[\\/]", "", regex=True).str.replace(r"\.(jpg|jpeg|png)$", "", regex=True).values

def load_hybrid_data():
    # STRICT ISOLATION: Only train and validation splits are combined; the internal test split is excluded.
    df_tr = pd.read_csv(os.path.join(SPLIT_DIR, "train_split.csv"))
    df_va = pd.read_csv(os.path.join(SPLIT_DIR, "val_split.csv"))
    df_int = pd.concat([df_tr, df_va], ignore_index=True)
    ids_int = _normalize_ids(df_int["image_id" if "image_id" in df_int.columns else "image"])
    y_int = df_int["label"].astype(int).values if "label" in df_int.columns else df_int["class_name"].apply(lambda x: CANONICAL_CLASS_MAP.get(x.upper(), 0)).values

    df_ext = pd.read_csv(EXT_CSV)
    ids_ext = _normalize_ids(df_ext["image_id" if "image_id" in df_ext.columns else "image"])
    y_ext = df_ext["target_idx"].astype(int).values if "target_idx" in df_ext.columns else df_ext["label"].astype(int).values

    data = {}
    for feat in ["SWIN", "EVA", "COAT"]:
        p = Path(FEATURE_DIRS[feat])
        X1 = np.load(_glob_any(p, ["train_features.npy", "features_train.npy", "train.npy"]))
        X2 = np.load(_glob_any(p, ["val_features.npy", "features_val.npy", "val.npy"]))
        Xext = np.load(EXT_FEATURES[feat])
        data[feat] = {"Xint": np.vstack([X1, X2]), "Xext": Xext}
    return ids_int, y_int, ids_ext, y_ext, data

def load_combined_meta(csv_int, csv_ext):
    def ext(df):
        id_col = "image_id" if "image_id" in df.columns else "image"
        df["_img"] = _normalize_ids(df[id_col])
        age_s = df["age"] if "age" in df.columns else df.get("age_approx", pd.Series([np.nan]*len(df)))
        df["_age"] = pd.to_numeric(age_s, errors="coerce").clip(lower=0)

        reg_s = df["anatom_site_general"] if "anatom_site_general" in df.columns else df.get("region", pd.Series(["missing"]*len(df)))
        reg = reg_s.astype(str).str.strip().str.lower().replace({"nan": "missing", "none": "missing"})
        df["_region_raw"] = reg.where(~reg.isin(["", "nan", "none", "na"]), "missing")

        les_col = "lesion_id" if "lesion_id" in df.columns else "lesion"
        df["_lesion_id"] = df[les_col].astype(str) if les_col in df.columns else df["_img"]
        return df[["_img", "_age", "_region_raw", "_lesion_id"]]
    return pd.concat([ext(pd.read_csv(csv_int)), ext(pd.read_csv(csv_ext))], ignore_index=True).drop_duplicates("_img", keep="first")

def make_region_mapper(ids, meta):
    mp = meta.set_index("_img")["_region_raw"].to_dict()
    regs = pd.Series([mp.get(i, "missing") for i in ids])
    vocab = sorted(regs.value_counts().index.tolist() + ["missing"])

    # External-data handling: map unseen anatomical regions to 'missing'.
    def safe_mapper(id_list):
        res = []
        for i in id_list:
            r = mp.get(i, "missing")
            res.append(r if r in vocab else "missing")
        return pd.Series(res).to_dict()

    return vocab, safe_mapper

def augment(base_X, ids, meta_df, vocab, id2map, use_meta):
    if not use_meta: return base_X.astype(np.float32)
    age_mp = meta_df.set_index("_img")["_age"].to_dict()
    age_rows = []
    for img in ids:
        a = age_mp.get(img, np.nan)
        one = np.zeros(len(CHAMPION_AGE_CFG["edges"]) + (1 if CHAMPION_AGE_CFG["add_missing"] else 0), dtype=np.float32)
        if not np.isnan(a):
            for i, (lo, hi) in enumerate(CHAMPION_AGE_CFG["edges"]):
                if lo <= a < hi: one[i]=1.0; break
        age_rows.append(one)
    reg_block = np.zeros((len(ids), len(vocab)), dtype=np.float32)
    for r, img in enumerate(ids): reg_block[r, vocab.index(id2map.get(img, "missing"))] = 1.0
    return np.hstack([base_X, np.stack(age_rows), reg_block]).astype(np.float32)


## Classifiers and feature preprocessing

Defines the MLP classifier, inference helper, and train-fitted scaling/PCA pipeline.

In [ ]:

# -------------------- Models --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hp):
        super().__init__()
        layers = [nn.Linear(in_dim, hp["hidden"][0]), nn.BatchNorm1d(hp["hidden"][0]), nn.ReLU(), nn.Dropout(hp["dropout"]), nn.Linear(hp["hidden"][0], n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_mlp_and_predict(Xtr, ytr, Xval, Xte, hp, device):
    mdl = RobustMLP(Xtr.shape[1], 8, hp).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

    # EXACT PIPELINE: Standard CE with label smoothing
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)

    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    loader = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long)), batch_size=MLP_BATCH, shuffle=True, generator=g)

    mdl.train()
    for _ in range(hp["epochs"]):
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True); crit(mdl(xb.to(device)), yb.to(device)).backward(); opt.step()

    def infer(X):
        mdl.eval()
        with torch.no_grad(): return torch.softmax(mdl(torch.tensor(X, dtype=torch.float32).to(device)), 1).cpu().numpy()
    return infer(Xval), infer(Xte)

def pca_pipe(Xtr, Xva, Xte):
    sc = StandardScaler().fit(Xtr)
    Ztr, Zva, Zte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    pca_t = PCA(n_components=min(Ztr.shape)-1, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    k = np.argmax(np.cumsum(pca_t.explained_variance_ratio_) >= PCA_VARIANCE) + 1
    pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)


## Calibration, fusion, and decision-threshold optimization

Defines temperature scaling, validation-NLL calibration, NLL-based ensemble-weight search, and the OOF balanced-accuracy decision rule.

In [ ]:

# -------------------- Calibration & OOF Optimization --------------------
def temp_scale(P, T):
    log_Q = np.log(np.clip(P, 1e-15, 1.0)) / T
    Q = np.exp(log_Q - np.max(log_Q, axis=1, keepdims=True))
    return Q / Q.sum(axis=1, keepdims=True)

def pick_T_by_nll(y_true, P, n_class, step):
    grid = np.arange(T_MIN, T_MAX+1e-9, step)
    losses = [log_loss(y_true, temp_scale(P, t), labels=range(n_class)) for t in grid]
    return grid[np.argmin(losses)]

def weight_search_nll(P_dict, y_true, n_class):
    # FUSION STRICTLY BY NLL
    best = {"score": 1e18, "ws":0.33, "we":0.33, "wc":0.34}
    for ws in np.arange(0.0, 1.01, W_STEP):
        for we in np.arange(0.0, 1.01, W_STEP):
            wc = 1.0 - ws - we
            if wc < COAT_FLOOR - 1e-9: continue
            score = log_loss(y_true, np.clip(ws*P_dict["SWIN"] + we*P_dict["EVA"] + wc*P_dict["COAT"], 1e-15, 1.0), labels=range(n_class))
            if score < best["score"]: best.update({"score":score, "ws":ws, "we":we, "wc":wc})
    return best

def optimize_thresholds_bacc(y_true, P, n_classes, max_iter=5):
    # DECISION RULE STRICTLY BY BACC
    best_thrs = np.ones(n_classes)
    best_score = 0.0
    for _ in range(max_iter):
        improved = False
        for c in range(n_classes):
            for t in np.linspace(0.05, 5.0, 100):
                temp_thrs = best_thrs.copy()
                temp_thrs[c] = t
                y_pred = (P / temp_thrs).argmax(axis=1)
                score = balanced_accuracy_score(y_true, y_pred)
                if score > best_score:
                    best_score, best_thrs, improved = score, temp_thrs, True
        if not improved: break
    return best_thrs


## Metrics and bootstrap evaluation

Defines calibration/classification metrics, per-class reporting, and class-stratified bootstrap confidence intervals.

In [ ]:

# -------------------- Reporting Utilities (Separate Objectives) --------------------
def compute_ece(y_true, P):
    conf = np.max(P, 1); acc = (np.argmax(P, 1) == y_true)
    bins = np.linspace(0, 1, N_CALIB_BINS+1)
    return sum(np.abs(np.mean(conf[m]) - np.mean(acc[m])) * np.mean(m) for i in range(N_CALIB_BINS) if np.sum(m := ((conf > bins[i]) & (conf <= bins[i+1])) if i>0 else ((conf >= 0) & (conf <= bins[1]))) > 0)

def apply_rule(P, thrs=None):
    return P.argmax(1) if thrs is None else (P / thrs).argmax(1)

def eval_prob_metrics(y_true, P):
    P_c = np.clip(P, 1e-15, 1.0); P_c /= P_c.sum(1, keepdims=True)
    Y_ohe = np.eye(8)[y_true]
    return {
        "ROC_AUC": float(roc_auc_score(Y_ohe, P_c, average="macro", multi_class="ovr")),
        "ECE": compute_ece(y_true, P_c),
        "NLL": float(log_loss(y_true, P_c, labels=range(8))),
        "Brier": float(np.mean(np.sum((P_c - Y_ohe) ** 2, axis=1)))
    }

def eval_class_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=range(8))
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=range(8), zero_division=0)
    specs = np.zeros(8)
    for k in range(8):
        tp = cm[k, k]
        fp = cm[:, k].sum() - tp
        fn = cm[k, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs[k] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Macro_F1": float(np.mean(f1)),
        "Weighted_F1": float(np.sum(f1 * (sup / np.maximum(sup.sum(), 1)))),
        "Macro_Sensitivity": float(np.mean(rec)),
        "Macro_Specificity": float(np.mean(specs)),
        "Macro_Precision": float(np.mean(prec))
    }

def build_per_class_report(y_true, P, y_pred, class_names):
    n_class = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_class)))
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(n_class)), zero_division=0)

    Y_ohe = np.eye(n_class)[y_true]
    P_c = np.clip(P, 1e-15, 1.0)
    P_c /= P_c.sum(axis=1, keepdims=True)

    specs = np.zeros(n_class)
    roc_aucs = np.zeros(n_class)

    for k in range(n_class):
        tp = cm[k, k]
        fp = cm[:, k].sum() - tp
        fn = cm[k, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs[k] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        roc_aucs[k] = float(roc_auc_score(Y_ohe[:, k], P_c[:, k]))

    rows = [{"Class": cname, "ROC_AUC": float(roc_aucs[k]), "Precision": float(prec[k]),
             "Sensitivity": float(rec[k]), "Specificity": float(specs[k]),
             "F1_Score": float(f1[k]), "Support": int(sup[k])} for k, cname in enumerate(class_names)]

    w = sup / np.maximum(sup.sum(), 1)
    rows.append({"Class": "Macro_Avg", "ROC_AUC": float(roc_auc_score(Y_ohe, P_c, average="macro", multi_class="ovr")),
                 "Precision": np.mean(prec), "Sensitivity": np.mean(rec),
                 "Specificity": np.mean(specs), "F1_Score": np.mean(f1), "Support": int(sup.sum())})
    rows.append({"Class": "Weighted_Avg", "ROC_AUC": float(roc_auc_score(Y_ohe, P_c, average="weighted", multi_class="ovr")),
                 "Precision": np.sum(prec*w), "Sensitivity": np.sum(rec*w),
                 "Specificity": np.sum(specs*w), "F1_Score": np.sum(f1*w), "Support": int(sup.sum())})
    return pd.DataFrame(rows)

def bootstrap_full_evaluation(y_true, P, thrs_bacc, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(RANDOM_STATE)
    Pc = np.clip(P, 1e-15, 1.0); Pc /= Pc.sum(1, keepdims=True)
    idx_list = [np.where(y_true == k)[0] for k in range(8)]

    pt_prob = eval_prob_metrics(y_true, Pc)
    pt_std = eval_class_metrics(y_true, apply_rule(Pc, None))
    pt_opt_bacc = eval_class_metrics(y_true, apply_rule(Pc, thrs_bacc))
    pt_delta_bacc = pt_opt_bacc["Balanced_Accuracy"] - pt_std["Balanced_Accuracy"]

    boots = {"prob": {k: np.zeros(n_boot) for k in pt_prob}, "std": {k: np.zeros(n_boot) for k in pt_std},
             "opt_bacc": {k: np.zeros(n_boot) for k in pt_opt_bacc}, "delta_bacc": np.zeros(n_boot)}

    for b in range(n_boot):
        idx = np.concatenate([rng.choice(c_idx, size=len(c_idx), replace=True) for c_idx in idx_list if len(c_idx)>0])
        yb, pb = y_true[idx], Pc[idx]

        m_prob = eval_prob_metrics(yb, pb)
        m_std = eval_class_metrics(yb, apply_rule(pb, None))
        m_bacc = eval_class_metrics(yb, apply_rule(pb, thrs_bacc))

        for k in pt_prob: boots["prob"][k][b] = m_prob[k]
        for k in pt_std: boots["std"][k][b], boots["opt_bacc"][k][b] = m_std[k], m_bacc[k]
        boots["delta_bacc"][b] = m_bacc["Balanced_Accuracy"] - m_std["Balanced_Accuracy"]

    def fmt(pt, bt, rule):
        rows = []
        for k, p in pt.items():
            arr = bt[k][~np.isnan(bt[k])]
            lo, hi = (np.percentile(arr, 2.5), np.percentile(arr, 97.5)) if len(arr)>0 else (np.nan, np.nan)
            rows.append({"Decision Rule": rule, "Metric": k, "Estimate": p, "CI95_Low": lo, "CI95_High": hi, "Formatted_95CI": f"{p:.4f} [{lo:.4f}, {hi:.4f}]"})
        return rows

    res_prob = fmt(pt_prob, boots["prob"], "Decision-Free (Probabilities)")
    res_class = fmt(pt_std, boots["std"], "1. Standard Argmax") + fmt(pt_opt_bacc, boots["opt_bacc"], "2. BAcc-Optimized Rule")

    arr_delta = boots["delta_bacc"][~np.isnan(boots["delta_bacc"])]
    lo, hi = (np.percentile(arr_delta, 2.5), np.percentile(arr_delta, 97.5)) if len(arr_delta)>0 else (np.nan, np.nan)
    res_class.append({"Decision Rule": "Comparison", "Metric": "Delta_BAcc", "Estimate": pt_delta_bacc, "CI95_Low": lo, "CI95_High": hi, "Formatted_95CI": f"{pt_delta_bacc:+.4f} [{lo:+.4f}, {hi:+.4f}]"})

    return pd.DataFrame(res_class), pd.DataFrame(res_prob)


## External-test experiment loop

Runs the pre-specified VISUAL_ONLY and VISUAL_META experiments using grouped cross-validation on internal train+validation data, then evaluates the averaged fold ensemble on the official external test set.

In [ ]:

# -------------------- Main Experiment Loop --------------------
def main():
    set_seed()
    ids_int, y_int, ids_ext, y_ext, data = load_hybrid_data()
    meta_df = load_combined_meta(META_CSV, EXT_CSV)

    for exp_mode in EXPERIMENT_MODES:
        use_meta = (exp_mode == "VISUAL_META")
        exp_dir = os.path.join(BASE_SAVE_DIR, exp_mode)
        os.makedirs(exp_dir, exist_ok=True)
        print(f"\n{'='*80}\n🚀 STARTING: {exp_mode} (NLL Fusion + BAcc Decision)\n{'='*80}")

        region_vocab, map_ids = make_region_mapper(ids_int, meta_df)
        idmap_int, idmap_ext = map_ids(ids_int), map_ids(ids_ext)
        lesion_ids = meta_df.set_index("_img")["_lesion_id"].reindex(ids_int).values

        OOF_P = np.zeros((len(y_int), 8))
        EXT_P_Folds = np.zeros((N_SPLITS, len(y_ext), 8))

        sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

        for fold, (tr_idx, va_idx) in enumerate(sgkf.split(ids_int, y_int, groups=lesion_ids)):
            print(f"\n   🔄 [FOLD {fold+1}/{N_SPLITS}] Training & Calibration...")
            ids_tr, ids_va = ids_int[tr_idx], ids_int[va_idx]
            y_tr, y_va = y_int[tr_idx], y_int[va_idx]

            P_val, P_ext, T_sel = {}, {}, {}
            for m in ["SWIN", "EVA", "COAT"]:
                Xtr = augment(data[m]["Xint"][tr_idx], ids_tr, meta_df, region_vocab, idmap_int, use_meta)
                Xva = augment(data[m]["Xint"][va_idx], ids_va, meta_df, region_vocab, idmap_int, use_meta)
                Xte = augment(data[m]["Xext"], ids_ext, meta_df, region_vocab, idmap_ext, use_meta)

                Ztr, Zva, Zte = pca_pipe(Xtr, Xva, Xte)

                if m in ["SWIN", "EVA"]:
                    P_va_raw, P_te_raw = train_mlp_and_predict(Ztr, y_tr, Zva, Zte, NAMED_PRESETS[f"{m.capitalize()}MLP"], DEVICE)
                    T_i = pick_T_by_nll(y_va, P_va_raw, 8, step=T_STEP)
                    P_val[m], P_ext[m], T_sel[m] = temp_scale(P_va_raw, T_i), temp_scale(P_te_raw, T_i), T_i
                else:
                    best_nll, best_Pva, best_Pte, best_T = 1e18, None, None, None
                    for C in COAT_C_GRID:
                        lg = LogisticRegression(C=float(C), **{k:v for k,v in HP_LOGREG.items() if k!="C"}).fit(Ztr, y_tr)
                        P_va_raw, P_te_raw = lg.predict_proba(Zva), lg.predict_proba(Zte)
                        Tc = pick_T_by_nll(y_va, P_va_raw, 8, step=T_STEP)
                        if (nll := log_loss(y_va, np.clip(temp_scale(P_va_raw, Tc), 1e-15, 1.0))) < best_nll:
                            best_nll, best_Pva, best_Pte, best_T = nll, temp_scale(P_va_raw, Tc), temp_scale(P_te_raw, Tc), Tc
                    P_val["COAT"], P_ext["COAT"], T_sel["COAT"] = best_Pva, best_Pte, best_T

            # NLL FUSION OOF
            w = weight_search_nll(P_val, y_va, 8)
            P_ens_va = w["ws"]*P_val["SWIN"] + w["we"]*P_val["EVA"] + w["wc"]*P_val["COAT"]
            T_ens = pick_T_by_nll(y_va, P_ens_va, 8, step=STAGE2_T_STEP)

            OOF_P[va_idx] = temp_scale(P_ens_va, T_ens)
            EXT_P_Folds[fold] = temp_scale(w["ws"]*P_ext["SWIN"] + w["we"]*P_ext["EVA"] + w["wc"]*P_ext["COAT"], T_ens)

        print("\n[STEP 3] Optimizing the decision rule on global OOF predictions (maximum threshold: 5.0)...")
        opt_thrs_bacc = optimize_thresholds_bacc(y_int, OOF_P, 8)
        print(f"   -> OOF BAcc Rule \u03C4: {[round(t,3) for t in opt_thrs_bacc]}")

        print("\n[STEP 4] Ensembling external-test predictions and generating detailed reports...")
        P_final_ext = np.mean(EXT_P_Folds, axis=0)

        tab1_class, tab4_prob = bootstrap_full_evaluation(y_ext, P_final_ext, opt_thrs_bacc, N_BOOTSTRAP)
        tab_cls_std = build_per_class_report(y_ext, P_final_ext, apply_rule(P_final_ext, None), CANONICAL_CLASS_NAMES)
        tab_cls_opt = build_per_class_report(y_ext, P_final_ext, apply_rule(P_final_ext, opt_thrs_bacc), CANONICAL_CLASS_NAMES)

        tab1_class.to_csv(os.path.join(exp_dir, f"Table1_ClassificationRules_{exp_mode}.csv"), index=False)
        tab4_prob.to_csv(os.path.join(exp_dir, f"Table4_ProbCalibration_{exp_mode}.csv"), index=False)
        tab_cls_std.to_csv(os.path.join(exp_dir, f"Table2_Classwise_Argmax_{exp_mode}.csv"), index=False)
        tab_cls_opt.to_csv(os.path.join(exp_dir, f"Table2_Classwise_OptBAcc_{exp_mode}.csv"), index=False)
        np.save(os.path.join(exp_dir, f"y_proba_external.npy"), P_final_ext)
        np.save(os.path.join(exp_dir, f"y_proba_OOF_internal.npy"), OOF_P)

        print(f"\n--- [TABLE 1: EXTERNAL-TEST DECISION RULES ({exp_mode})] ---")
        print(tab1_class.groupby(["Decision Rule", "Metric"])[["Formatted_95CI"]].first().to_string())
        print(f"\n{exp_mode} completed successfully. Outputs -> {exp_dir}")

if __name__ == "__main__":
    main()
